# JGLUE Evaluation with lm-evaluation-harness

このノートブックは [EleutherAI lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) を使って **JGLUE** (Japanese General Language Understanding Evaluation) のタスクを評価するためのものです。

### 対象タスク (JGLUE 4タスク)

> **重要**: lm-eval v0.4.6 以降では、JGLUE タスクは `japanese_leaderboard` 配下に登録されています。ベア ID (`jcommonsenseqa` 等) は使えないので注意してください。

| Task ID | タスク種別 | データセット | 評価指標 | few-shot (デフォルト) |
|---|---|---|---|---|
| `ja_leaderboard_jcommonsenseqa` | 多肢選択 | JCommonsenseQA | Accuracy | 3 |
| `ja_leaderboard_jnli` | 多肢選択 | JNLI | Accuracy | 3 |
| `ja_leaderboard_jsquad` | 抽出型QA (generate_until) | JSQuAD | Exact Match | 2 |
| `ja_leaderboard_marc_ja` | 多肢選択 (感情分析・2値) | MARC-ja | Accuracy | 3 |

### 必要な環境
- Google Colab (GPU ランタイム推奨。T4/A100/L4 など)
- Hugging Face Hub からモデルをダウンロードできること

### 使い方
1. 上から順番にセルを実行してください
2. 「設定」セルで評価対象のモデル名やバッチサイズを変更できます
3. 結果は `./results` に JSON で保存されます。Google Drive にバックアップすることも可能です


## 1. 依存ライブラリのインストール

lm-evaluation-harness と Hugging Face 関連ライブラリをインストールします。

> **NOTE**: Colab のプリインストール transformers とバージョン衝突することがあるため、`-q` で静かにインストールします。警告が出ても問題なければ次に進んでください。


In [ ]:
# lm-evaluation-harness のインストール
# pip 版 (安定) を使います。最新機能を使いたい場合は git+https://... に切り替えてください。
!pip install -q "lm-eval>=0.4.6" \
                "transformers>=4.43" \
                "accelerate>=0.30" \
                "datasets>=2.19" \
                "sentencepiece" \
                "protobuf"


In [ ]:
# バージョン確認
import lm_eval, transformers, torch, sys
print(f"Python       : {sys.version.split()[0]}")
print(f"lm-eval      : {lm_eval.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"torch        : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. 評価設定

以下のセルで評価対象モデルやバッチサイズなどを設定します。
デフォルトは [rinna/japanese-gpt2-small](https://huggingface.co/rinna/japanese-gpt2-small) (100M params) で、Colab 無料枠の T4 でも動作確認できます。

### 推奨モデル例
| モデル | パラメータ数 | 想定GPU |
|---|---|---|
| `rinna/japanese-gpt2-small` | 100M | T4 (Free) |
| `line-corporation/japanese-large-lm-1.7b` | 1.7B | T4 / A10G |
| `elyza/ELYZA-japanese-Llama-2-7b-instruct` | 7B | A100 (40GB) |
| `tokyotech-llm/Swallow-7b-instruct-v0.1` | 7B | A100 (40GB) |
| `mistralai/Mistral-7B-v0.3` | 7B | A100 (40GB) |


In [ ]:
# === 評価設定 ===
# ここを書き換えてください

MODEL_NAME = "rinna/japanese-gpt2-small"   # Hugging Face の model id
# JGLUE 4タスク (lm-eval v0.4.6+ では ja_leaderboard_* プレフィックスが必要)
TASKS = (
    "ja_leaderboard_jcommonsenseqa,"
    "ja_leaderboard_jnli,"
    "ja_leaderboard_jsquad,"
    "ja_leaderboard_marc_ja"
)
# バッチサイズ。"auto" にすると VRAM に収まる最大サイズを自動探索します (推奨)。
# 数値を指定すると固定 (例: 8)。OOM が出る場合は "auto:4" のように上限を設定できます。
BATCH_SIZE = "auto"
# few-shot 数。None にするとタスク yaml のデフォルト (JNLI/JCommonsenseQA/MARC=3, JSQuAD=2) を使います。
# JGLUE 論文準拠の 0-shot 評価をする場合は 0 を指定してください。
NUM_FEW_SHOT = 0
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "./results"
TRUST_REMOTE_CODE = False                   # モデルがカスタムコードを要求する場合は True

# サンプル数を制限する場合は整数 (例: 100)。None で全件評価。
# 全件評価するとモデルサイズによっては数時間かかります。
LIMIT = None

print(f"Model       : {MODEL_NAME}")
print(f"Tasks       : {TASKS}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Device      : {DEVICE}")
print(f"Few-shot    : {NUM_FEW_SHOT}")
print(f"Limit       : {LIMIT}")


In [ ]:
# Hugging Face にログイン (gated model を使う場合)
# 対話的ログイン:
# from huggingface_hub import login
# login()

# または Colab Secret を使う場合:
# import os
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


## 3. JGLUE タスクの確認

lm-eval に JGLUE タスクが登録されているか確認します。


In [ ]:
# JGLUE 関連タスクを一覧表示
# lm-eval v0.4.6+ では japanese_leaderboard 配下に JGLUE タスクが登録されています
!lm_eval --tasks list 2>/dev/null | grep -iE "ja_leaderboard_(jcommonsenseqa|jnli|jsquad|marc_ja)"


## 4. 評価の実行

`lm_eval` CLI を呼び出して評価を実行します。

> **NOTE**: 全タスク・全件評価すると、モデルサイズにもよりますが数時間かかることがあります。まずは `LIMIT = 100` などで煙テストすることを推奨します。

> **NOTE**: `ja_leaderboard_jsquad` は `generate_until` 型タスクです。GPT-2 系の非インストラクトモデルでは EM が 0 になりがちです。抽出型QAが期待通り動くか、まずは少数サンプル (`LIMIT=20`) で確認することを推奨します。


In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# 評価コマンドを組み立て
cmd_parts = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={MODEL_NAME},trust_remote_code={TRUST_REMOTE_CODE}",
    "--tasks", TASKS,
    "--device", DEVICE,
    "--batch_size", str(BATCH_SIZE),
    "--output_path", OUTPUT_DIR,
    "--log_samples",
]
# few-shot 数の指定。
# - 0 を指定すると 0-shot 評価 (JGLUE 論文準拠)
# - None の場合はタスク yaml デフォルト (JNLI/JCommonsenseQA/MARC=3, JSQuAD=2)
if NUM_FEW_SHOT is not None:
    cmd_parts += ["--num_fewshot", str(NUM_FEW_SHOT)]
if LIMIT is not None:
    cmd_parts += ["--limit", str(LIMIT)]

# シェルに渡すためにタスク文字列をクォート (カンマを含むため)
cmd = " ".join(cmd_parts)
print("Run command:\n")
print(cmd)
print("\n--- 評価を開始します (時間がかかります) ---\n")


In [ ]:
# 評価を実行
!{cmd}


## 5. 結果の確認

`./results` に保存された JSON を読み込んで結果を表示します。


In [ ]:
import json, glob, os
from pathlib import Path

result_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/results.json", recursive=True))
if not result_files:
    # 古い lm-eval は別のファイル名で保存することがある
    result_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/*.json", recursive=True))

print(f"Found {len(result_files)} result file(s):")
for p in result_files:
    print(f"  - {p}")

assert result_files, "結果ファイルが見つかりません。上の評価セルを再実行してください。"
latest = result_files[-1]
with open(latest) as f:
    data = json.load(f)


In [ ]:
# 結果を表形式で表示
from IPython.display import display, Markdown

rows = []
for task_name, task_metrics in data.get("results", {}).items():
    for metric_name, metric_val in task_metrics.items():
        if isinstance(metric_val, (int, float)) and "stderr" not in metric_name and "_se" not in metric_name:
            rows.append((task_name, metric_name, metric_val))

if rows:
    md = "| Task | Metric | Value |\n|---|---|---|\n"
    for t, m, v in rows:
        md += f"| {t} | {m} | {v*100:.2f}% |\n" if v <= 1.0 else f"| {t} | {m} | {v:.4f} |\n"
    display(Markdown(md))
else:
    print("結果キー:", list(data.get("results", {}).keys()))
    print(json.dumps(data.get("results", {}), indent=2, ensure_ascii=False))


In [ ]:
# メタ情報 (モデル・CLI・環境)
print("=== Configuration ===")
cfg = data.get("config", {})
print(json.dumps(cfg, indent=2, ensure_ascii=False, default=str)[:2000])


## 6. (Optional) 結果を Google Drive にバックアップ

評価結果を Google Drive に保存します。任意で実行してください。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# 保存先
DRIVE_DIR = "/content/drive/MyDrive/lm_eval_results"
os.makedirs(DRIVE_DIR, exist_ok=True)

# 全結果ファイルをコピー
import shutil
for p in result_files:
    dst = os.path.join(DRIVE_DIR, os.path.basename(os.path.dirname(p)) + "_" + os.path.basename(p))
    shutil.copy(p, dst)
    print(f"copied -> {dst}")


## 7. (Optional) vLLM バックエンドを使う (高速化)

大規模モデル (7B+) を評価する場合は vLLM バックエンドを使うと 5-10x 高速化します。

```python
# インストール
# !pip install -q vllm

# 評価コマンド例
# !lm_eval --model vllm \
#   --model_args pretrained=elyza/ELYZA-japanese-Llama-2-7b-instruct,dtype=bfloat16,gpu_memory_utilization=0.9 \
#   --tasks {TASKS} \
#   --batch_size auto \
#   --output_path {OUTPUT_DIR}
# ```


## 8. トラブルシューティング

| 症状 | 原因 / 対処 |
|---|---|
| `TaskNotFound` | `lm_eval --tasks list` で正確なタスク名を確認。lm-eval v0.4.6+ では `ja_leaderboard_*` プレフィックス必須 |
| `lm-eval` のバージョンが古い | `!pip install -U "lm-eval>=0.4.6"` で更新 |
| `OOM (Out Of Memory)` | `BATCH_SIZE = "auto:4"` のように上限を設定、またはより小さいモデルを使う |
| `trust_remote_code` エラー | モデルがカスタムコードを要求。`TRUST_REMOTE_CODE = True` に設定 |
| `ConnectionError` for HF Hub | Colab のネットワーク再接続、または `HF_HUB_OFFLINE=0` 確認 |
| `jsquad` で EM が常に 0 | `generate_until` 型タスクです。非インストラクトモデル (gpt2 等) では指示フォーマットに追従できないため。インストラクトモデルを使うか `--gen_kwargs do_sample=False` を追加 |
| MARC-ja タスクが見つからない | `ja_leaderboard_marc_ja` (プレフィックス + アンダースコア) を使ってください |


## 9. 参考

- **lm-evaluation-harness**: https://github.com/EleutherAI/lm-evaluation-harness
- **JGLUE paper**: Kurihara et al., "JGLUE: Japanese General Language Understanding Evaluation", ACL 2022. https://aclanthology.org/2022.acl-long.172/
- **JGLUE HF dataset**: https://huggingface.co/datasets/Rakuten/JGLUE
- **各タスクの yaml 設定**: https://github.com/EleutherAI/lm-evaluation-harness/tree/main/lm_eval/tasks/japanese_leaderboard
- **Japanese Leaderboard 追加 PR (#2439)**: https://github.com/EleutherAI/lm-evaluation-harness/pull/2439
